In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# DATA CLEANING & STANDARDIZATION

# Find the project folder
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_folder = current_folder.parent
else:
    project_folder = current_folder

data_folder = project_folder / "data"

if not data_folder.exists():
    raise FileNotFoundError(
        f"Data folder not found at: {data_folder}\n"
        "Make sure the notebook is inside the project folder."
    )

# 1. Find the original CSV

# Ignore previously generated cleaned files
csv_files = [
    file for file in data_folder.glob("*.csv")
    if "cleaned" not in file.stem.lower()
]

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No original CSV file found inside: {data_folder}"
    )

if len(csv_files) > 1:
    print("CSV files found:")
    for file in csv_files:
        print(" -", file.name)

    raise ValueError(
        "More than one original CSV file was found. "
        "Keep only the actual raw dataset in the data folder."
    )

csv_file = csv_files[0]

# Load the original dataset
df = pd.read_csv(csv_file)

print("DATA CLEANING & STANDARDIZATION")

print("\nOriginal dataset shape:")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# Keep a copy of the original data in memory
clean_df = df.copy()


# 2. Clean column names

print("COLUMN NAMES")

print("Original column names:")
print(list(clean_df.columns))

# Remove extra spaces and make column names easier to work with
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Make a few names clearer and consistent
clean_df = clean_df.rename(
    columns={
        "rate_per_sqft": "rate_per_sqft",
        "bhk_count": "bhk_count",
        "property_type": "property_type",
        "builder_name": "builder_name",
        "rera_approval": "rera_approval",
        "company_name": "company_name",
        "flat_type": "flat_type"
    }
)

print("\nCleaned column names:")
print(list(clean_df.columns))


# 3. Remove leading/trailing spaces from text columns

print("TEXT CLEANING")

# Use string dtype explicitly to avoid the pandas object/string warning
text_columns = clean_df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Text columns:")
print(text_columns)

for column in text_columns:
    clean_df[column] = clean_df[column].astype("string").str.strip()

print("\nLeading/trailing spaces have been removed.")


# 4. Convert Price to numeric

print("PRICE CONVERSION")

price_before = clean_df["price"].copy()

# Remove commas and spaces before conversion
clean_df["price"] = (
    clean_df["price"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
)

clean_df["price"] = pd.to_numeric(
    clean_df["price"],
    errors="coerce"
)

price_invalid = clean_df["price"].isna().sum()

print(f"Invalid Price values after conversion: {price_invalid}")

print("\nPrice data type:")
print(clean_df["price"].dtype)

print("\nPrice summary:")
print(clean_df["price"].describe())


# 5. Convert Rate per sqft to numeric

print("RATE PER SQFT CONVERSION")

rate_before = clean_df["rate_per_sqft"].copy()

# Remove commas and spaces
clean_df["rate_per_sqft"] = (
    clean_df["rate_per_sqft"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
)

clean_df["rate_per_sqft"] = pd.to_numeric(
    clean_df["rate_per_sqft"],
    errors="coerce"
)

rate_invalid = clean_df["rate_per_sqft"].isna().sum()

print(f"Invalid Rate per sqft values after conversion: {rate_invalid}")

print("\nRate per sqft data type:")
print(clean_df["rate_per_sqft"].dtype)

print("\nRate per sqft summary:")
print(clean_df["rate_per_sqft"].describe())


# 6. Make sure Area and BHK_Count are numeric

print("NUMERIC COLUMN CHECK")

clean_df["area"] = pd.to_numeric(
    clean_df["area"],
    errors="coerce"
)

clean_df["bhk_count"] = pd.to_numeric(
    clean_df["bhk_count"],
    errors="coerce"
)

print("Numeric columns:")
print(clean_df[["price", "area", "rate_per_sqft", "bhk_count"]].dtypes)


# 7. Check for missing values created during conversion

print("MISSING VALUE CHECK AFTER CONVERSION")

missing_after_conversion = clean_df.isna().sum()

print(
    missing_after_conversion[
        missing_after_conversion > 0
    ]
)

if missing_after_conversion.sum() == 0:
    print("\nNo missing values were created during cleaning.")
else:
    print(
        "\nWarning: Some missing values were created during "
        "numeric conversion and need investigation."
    )


# 8. Check categorical values after whitespace cleaning

print("CATEGORY CHECK")

category_columns = [
    "status",
    "property_type",
    "locality",
    "builder_name",
    "rera_approval",
    "society",
    "company_name",
    "flat_type"
]

for column in category_columns:
    print(f"\n{column}")
    print(f"Unique values: {clean_df[column].nunique(dropna=False):,}")
    
    # Show the most common values
    print(clean_df[column].value_counts(dropna=False).head(10))


# 9. Check exact duplicate rows

print("DUPLICATE CHECK")

duplicates_before = clean_df.duplicated().sum()

print(f"Exact duplicate rows before removal: {duplicates_before:,}")

# Remove only exact duplicate rows
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

duplicates_after = clean_df.duplicated().sum()

print(f"Exact duplicate rows after removal: {duplicates_after:,}")

print(f"\nRows after duplicate removal: {len(clean_df):,}")
print(f"Rows removed: {duplicates_before:,}")


# 10. Check suspicious numeric values

print("SUSPICIOUS NUMERIC VALUE CHECK")

print("\nBHK values:")
print("Minimum:", clean_df["bhk_count"].min())
print("Maximum:", clean_df["bhk_count"].max())
print("Zero BHK rows:", (clean_df["bhk_count"] == 0).sum())
print("Negative BHK rows:", (clean_df["bhk_count"] < 0).sum())

print("\nArea values:")
print("Minimum:", clean_df["area"].min())
print("Maximum:", clean_df["area"].max())
print("Zero Area rows:", (clean_df["area"] == 0).sum())
print("Negative Area rows:", (clean_df["area"] < 0).sum())

print("\nRate per sqft values:")
print("Minimum:", clean_df["rate_per_sqft"].min())
print("Maximum:", clean_df["rate_per_sqft"].max())
print(
    "Zero Rate per sqft rows:",
    (clean_df["rate_per_sqft"] == 0).sum()
)
print(
    "Negative Rate per sqft rows:",
    (clean_df["rate_per_sqft"] < 0).sum()
)

print("\nPrice values:")
print("Minimum:", clean_df["price"].min())
print("Maximum:", clean_df["price"].max())
print(
    "Zero Price rows:",
    (clean_df["price"] == 0).sum()
)
print(
    "Negative Price rows:",
    (clean_df["price"] < 0).sum()
)


# 11. Create a cleaning summary

print("CLEANING SUMMARY")

print(f"Original rows:       {len(df):,}")
print(f"Cleaned rows:        {len(clean_df):,}")
print(f"Rows removed:        {len(df) - len(clean_df):,}")
print(f"Original columns:    {df.shape[1]}")
print(f"Cleaned columns:     {clean_df.shape[1]}")

print("\nData types after cleaning:")
print(clean_df.dtypes)


# 12. Final duplicate and missing-value check

print("FINAL QUALITY CHECK")

final_missing = clean_df.isna().sum().sum()
final_duplicates = clean_df.duplicated().sum()

print(f"Total missing values: {final_missing:,}")
print(f"Total duplicate rows: {final_duplicates:,}")

if final_missing == 0 and final_duplicates == 0:
    print("\nBasic cleaning checks passed.")
else:
    print("\nSome issues still need investigation.")


# 13. Save cleaned dataset

cleaned_file = data_folder / "propertylens_cleaned.csv"

clean_df.to_csv(
    cleaned_file,
    index=False
)

print("CLEANED DATASET SAVED")

print(f"File: {cleaned_file}")
print(f"Rows saved: {len(clean_df):,}")
print(f"Columns saved: {len(clean_df.columns)}")


# 14. Show sample of cleaned data

print("FIRST 5 ROWS OF CLEANED DATA")

print(clean_df.head().to_string(index=False))

DATA CLEANING & STANDARDIZATION

Original dataset shape:
Rows: 19,515
Columns: 12
COLUMN NAMES
Original column names:
['Price', 'Status', 'Area', 'Rate per sqft', 'Property Type', 'Locality', 'Builder Name', 'RERA Approval', 'BHK_Count', 'Society', 'Company Name', 'Flat Type']

Cleaned column names:
['price', 'status', 'area', 'rate_per_sqft', 'property_type', 'locality', 'builder_name', 'rera_approval', 'bhk_count', 'society', 'company_name', 'flat_type']
TEXT CLEANING
Text columns:
['price', 'status', 'rate_per_sqft', 'property_type', 'locality', 'builder_name', 'rera_approval', 'society', 'company_name', 'flat_type']

Leading/trailing spaces have been removed.
PRICE CONVERSION
Invalid Price values after conversion: 0

Price data type:
Int64

Price summary:
count            19515.0
mean     40286933.487061
std      59541101.252879
min              95000.0
25%           14500000.0
50%           26500000.0
75%           45000000.0
max         1226300000.0
Name: price, dtype: Float64
RA